# 04 · Evaluate on the sealed test set

In [ ]:
import os, sys
REPO_URL = "https://github.com/YUPWE/car-models.git"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/car-models"
    if not os.path.exists(ROOT):
        !git clone {REPO_URL} {ROOT}
    os.chdir(ROOT)
    !git pull -q
    !pip install -q imagehash==4.3.2
    os.environ["CARS_DATA_DIR"] = "/content/cars_data"
else:
    ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
    os.chdir(ROOT)
sys.path.insert(0, ROOT)
os.makedirs("results", exist_ok=True)
os.makedirs("weights", exist_ok=True)
QUICK = os.environ.get("QUICK", "0") == "1"
print("Working in", os.getcwd(), "| QUICK =", QUICK)

In [ ]:
import json, shutil, numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from src import data
from src.data import CarDataset, get_transforms
from src.model import build_resnet18, predict_probs, get_device
from src.clip_gate import ClipZeroShot, load_pil
from src.system import decide, mix, load_config, save_config

data.download()
device = get_device()
names = json.load(open("results/model_names.json"))
cfg = load_config(); W, OOS_T = cfg["w"], cfg["oos_threshold"]
print("from notebook 03:", cfg)
splits = {s: data.load_split(s) for s in ["val", "test"]}
if QUICK: splits = {s: d.sample(64, random_state=0).reset_index(drop=True) for s, d in splits.items()}

def load_model(path):
    m = build_resnet18(freeze_backbone=False, pretrained=False, num_classes=196)
    m.load_state_dict(torch.load(path, map_location=device)); return m.to(device)
models = {"frozen": load_model("weights/resnet18_frozen.pt"), "finetuned": load_model("weights/resnet18_finetuned.pt")}
clip = ClipZeroShot(names, device=device)
P = {}
for s, d in splits.items():
    loader = DataLoader(CarDataset(d, get_transforms(False)), batch_size=128, num_workers=2)
    P[s] = {}
    for k, m in models.items():
        P[s][k], P[s]["y"] = predict_probs(m, loader, device)
    P[s]["clip"], P[s]["oos"] = clip(load_pil(f"{data.IMG_DIR}/{p}" for p in d.path))
    P[s]["mixed"] = mix(P[s]["finetuned"], P[s]["clip"], W)

## 1 · Choose the auto-fill threshold (validation only)

In [ ]:
TARGET = 0.95
def coverage_curve(p, y, ts):
    conf, pred = p.max(1), p.argmax(1)
    cov = np.array([(conf >= t).mean() for t in ts])
    acc = np.array([(pred[conf >= t] == y[conf >= t]).mean() if (conf >= t).any() else np.nan for t in ts])
    return cov, acc
ts = np.round(np.arange(0.05, 1.0, 0.01), 2)
cov_v, acc_v = coverage_curve(P["val"]["mixed"], P["val"]["y"], ts)
good = ts[acc_v >= TARGET]
CONF_T = float(good.min()) if len(good) else 0.99
save_config(conf_threshold=CONF_T, target_accuracy=TARGET)
i = int(np.where(ts == CONF_T)[0][0])
print(f"chosen threshold = {CONF_T}: val auto-fill rate {cov_v[i]:.1%}, val accuracy on auto-filled {acc_v[i]:.1%}")
cov_t, acc_t = coverage_curve(P["test"]["mixed"], P["test"]["y"], ts)
plt.figure(figsize=(6, 4))
plt.plot(cov_v, acc_v, label="validation"); plt.plot(cov_t, acc_t, label="test")
plt.axhline(TARGET, ls=":", c="gray", label="target 95%")
plt.scatter([cov_v[i]], [acc_v[i]], c="k", zorder=5, label=f"chosen threshold {CONF_T}")
plt.xlabel("auto-fill rate (share of photos filled automatically)"); plt.ylabel("accuracy on auto-filled photos")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.savefig("results/coverage_curve.png", dpi=120); plt.show()

## 2 · Test set: model comparison

In [ ]:
yt = P["test"]["y"]
def topk(p, y, k): return float((np.argsort(-p, 1)[:, :k] == y[:, None]).any(1).mean())
rows = []
for name, key in [("CLIP zero-shot (no training)", "clip"), ("ResNet-18 frozen (baseline)", "frozen"),
                  ("ResNet-18 fine-tuned", "finetuned"), (f"Fine-tuned + CLIP mix (w={W})", "mixed")]:
    p = P["test"][key]
    rows.append({"model": name, "top-1 accuracy": topk(p, yt, 1), "top-3 accuracy": topk(p, yt, 3),
                 "top-5 accuracy": topk(p, yt, 5), "macro-F1": f1_score(yt, p.argmax(1), average="macro")})
table = pd.DataFrame(rows).round(3)
print(f"Test set: {len(yt)} images, {len(set(yt))} car models")
print(table.to_string(index=False)); table.to_csv("results/test_comparison.csv", index=False)

In [ ]:
pred = P["test"]["mixed"].argmax(1)
cm = confusion_matrix(yt, pred, labels=range(196)).astype(float)
cm = cm / cm.sum(1, keepdims=True).clip(min=1)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_title("Test confusion matrix, 196 models (rows = true, normalised)")
ax.set_xlabel("predicted model id"); ax.set_ylabel("true model id"); plt.colorbar(im, fraction=0.046)
plt.tight_layout(); plt.savefig("results/confusion_matrix.png", dpi=120); plt.show()

test_df = splits["test"].reset_index(drop=True).copy()
test_df["pred"] = [names[i] for i in pred]
wrong = test_df[test_df.model != test_df.pred]
pairs = wrong.groupby(["model", "pred"]).size().sort_values(ascending=False).reset_index(name="count").head(15)
pairs["same make"] = [data.make_of(a) == data.make_of(b) for a, b in zip(pairs.model, pairs.pred)]
print("Most confused pairs:"); print(pairs.to_string(index=False)); pairs.to_csv("results/top_confusions.csv", index=False)
same_make = np.mean([data.make_of(a) == data.make_of(b) for a, b in zip(wrong.model, wrong.pred)])
print(f"\n{len(wrong)} test mistakes; {same_make:.1%} of them are confused with a car of the SAME make")
by_make = test_df.assign(correct=test_df.model == test_df.pred).groupby("make").correct.agg(["count", "mean"]).sort_values("mean").round(3)
print("\nWorst makes:"); print(by_make.head(8).to_string()); by_make.to_csv("results/accuracy_by_make.csv")

## 3 · Test set: the full system

In [ ]:
dec, mixed_p = decide(P["test"]["finetuned"], P["test"]["clip"], P["test"]["oos"], W, CONF_T, OOS_T)
dec = np.array(dec); auto = dec == "auto-fill"
top3_hit = (np.argsort(-mixed_p, 1)[:, :3] == yt[:, None]).any(1)
summary = {
    "test images": len(dec),
    "auto-filled": f"{auto.mean():.1%}",
    "accuracy on auto-filled": f"{(pred[auto] == yt[auto]).mean():.1%}",
    "wrong auto-fills": int((pred[auto] != yt[auto]).sum()),
    "asked seller (top-3 shown)": f"{(dec == 'ask seller').mean():.1%}",
    "right model inside the top-3 shown to seller": f"{top3_hit[dec == 'ask seller'].mean():.1%}",
    "real cars wrongly called 'not a car'": f"{(dec == 'not a car').mean():.1%}",
}
for k, v in summary.items(): print(f"{k:46s} {v}")
json.dump(summary, open("results/system_test_summary.json", "w"), indent=1)

## 4 · Failure images

In [ ]:
shutil.rmtree("results/failures", ignore_errors=True); os.makedirs("results/failures")
conf = mixed_p.max(1)
wrong_idx = np.where(pred != yt)[0]; wrong_idx = wrong_idx[np.argsort(-conf[wrong_idx])][:12]
records = []
fig, axes = plt.subplots(2, 6, figsize=(20, 7)); axes = axes.ravel()
for k, i in enumerate(wrong_idx):
    row = test_df.iloc[i]
    fname = f"{k + 1:02d}_{data.make_of(row.model)}.jpg".replace(" ", "-")
    shutil.copy(f"{data.IMG_DIR}/{row.path}", f"results/failures/{fname}")
    records.append({"file": fname, "true": row.model, "predicted": names[pred[i]], "confidence": round(float(conf[i]), 3),
                    "decision": dec[i], "cause": "", "fix": ""})
    axes[k].imshow(plt.imread(f"{data.IMG_DIR}/{row.path}")); axes[k].axis("off")
    axes[k].set_title(f"true: {row.model[:30]}\npred: {names[pred[i]][:30]}\nconf {conf[i]:.2f} -> {dec[i]}", fontsize=8)
for a in axes[len(wrong_idx):]: a.axis("off")
plt.tight_layout(); plt.savefig("results/failures_overview.png", dpi=100); plt.show()
pd.DataFrame(records).to_csv("results/failures/failures.csv", index=False)
print(pd.DataFrame(records)[["file", "true", "predicted", "confidence"]].to_string(index=False))

## 5 · Numbers for the README

In [ ]:
print(table.to_markdown(index=False))
print(f"\n- Auto-fill threshold: {CONF_T} (lowest value with validation accuracy >= {TARGET:.0%})")
print(f"- Not-a-car threshold: {OOS_T} | CLIP mixing weight w: {W}")
for k, v in summary.items(): print(f"- {k}: {v}")